## Notebook to demo Point In Polygon analysis where polygons are in a shapefile.

In [ ]:
# from spark_esri import spark_start, spark_stop

In [1]:
import os

import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

In [2]:
# config = {
#     "spark.driver.memory":"32G",
#     "spark.executor.memory":"32G",
#     "spark.jars":"geofunctions-0.28.jar",
# }

# spark_stop()
# spark = spark_start(config=config)

### Get the number of QR parallel configuration.

In [3]:
# spark.conf.set("spark.esri.parallel", 1_000_000)

# spark.conf.get("spark.esri.accelerate")
spark.conf.get("spark.esri.parallel")

'100'

In [4]:
cell = 5.0  # QR in degrees.
dist = 0.00001  # Padding in degrees.
wkid = 4326  # SR ID

### Generate random points.

In [5]:
xmin, ymin, xmax, ymax = (-180.0, -80.0, 180.0, 80.0)
xdel = xmax - xmin
ydel = ymax - ymin

lhs = (
    spark.range(0, 10_000_000)
    .repartition(200)
    .withColumn("x", F.rand() * xdel + xmin)
    .withColumn("y", F.rand() * ydel + ymin)
    .select(
        S.st_point("x", "y").alias("pt"),
        S.qr_fromxy("x", "y", cell).alias("qr"),
    )
    .repartition(200, "qr")
    .localCheckpoint()
)

### Read the time zone polygons.

In [6]:
rhs = (
    spark.read.format("shp")
    .options(path="../data/world/tz_world.shp")
    .load()
    .select(
        "tzid",
        S.st_dump_explode("shape").alias("shape"),
    )
    .select(
        "tzid",
        S.qr_geom_explode(
            "shape",
            cell,
            dist,
            wkid,
        ).alias("qr"),
    )
    .select(
        "tzid",
        "qr.qr",
        "qr.geom",
    )
    .repartition(200, "qr")
    .localCheckpoint()
)

### Perform Point in Polygon.

In [7]:
%%time

df = (
    lhs.join(rhs, "qr")
    .drop("qr")
    .filter(S.st_contains("geom", "pt", wkid))
    .drop("geom")
    .localCheckpoint()
)

[Stage 13:=====================================================>(160 + 1) / 161]

CPU times: user 74.6 ms, sys: 39.4 ms, total: 114 ms
Wall time: 33.2 s


### Convert to GeoPandas and visualize.

In [8]:
fraction = 50_000 / 10_000_000

In [9]:
pdf = df.sample(fraction=fraction).withColumnRenamed("pt", "geometry").toPandas()
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs=f"EPSG:{wkid}")
# Enable the spatial index.
gdf.sindex
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)